# Importing Dependencies

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as sc

# Defining Variables

$T$: Time till maturity

$N$: Number of steps


$\Delta t$: Time step size, given by $\Delta t = \frac{T}{N}$

$\sigma$: Volatility of underlying asset

$u$: Multiplicative up factor (how much the price increases on a single time step), $u=e^{\sigma\sqrt{\Delta t}}$

$d$: Multiplicative down factor, $d=\frac{1}{u}$

$r$: Risk free interest rate

$p$: Risk neutral probability given by, $p =\frac{e^{r \cdot dt} - d}{u - d}$

$S_0$: Initial stock price

$K$: Strike price

In [2]:
T = 1
n = 252
dt = T/n
S0 = 100
K = 102
sigma = 0.2
u = np.exp(sigma*np.sqrt(dt))
d = 1/u
r = 0.03
p = (np.exp(r * dt)-d) / (u-d)

# The Cox-Ross-Rubinstein Model

Terminal stock price:

$$S_{N,j} = S_0 u^j  d^{N-j}$$

where $j\in[0,N]$

Call payoff:

$$C_{N,j} = \max(S_{N,j} - K, 0)$$

Put payoff:

$$P_{N,j} = \max(K - S_{N,j}, 0)$$

Node valuation:

$$V_{i,j} = e^{-r \Delta t} \left( p V_{i+1,j+1} + (1 - p) V_{i+1,j} \right)$$

where $i = N-1, N-2, ... , 0$ and $j \in [0,i]$. The price of the option is given by $V_{0,0}$ The initial values for $V_{i,j}$ are given by $C_{N,j}$ for calls and $P_{N,j}$ for puts.

In [3]:
def CRR(S0,K,u,d,r,p,n,dt,call):
  num_up = np.arange(0,n+1)
  St = S0*(u**num_up)*(d**(n-num_up))

  if call:
    value = np.maximum(St-K,0)
  else:
    value = np.maximum(K-St,0)

  for i in range(0,n,1):
    value = np.exp(-r * dt) * (p * value[1:] + (1-p) * value[:-1])

  return round(value[0],4)

# Pricing with Cox-Ross-Rubinstein

In [4]:
print(f"Call Price: ${CRR(S0,K,u,d,r,p,n,dt,True)}")
print(f"Put Price: ${CRR(S0,K,u,d,r,p,n,dt,False)}")

Call Price: $8.4451
Put Price: $7.4305


# Closed Form Cox-Ross-Rubinstein Call

Closed form formula:

$$C_0 = e^{-rT} \sum_{i=a}^{N} \left[ \binom{N}{i} p^i (1-p)^{N-i} (S_0 u^i d^{N-i} - K) \right]$$

Where $a=\min(j:S_{0}u^jd^{N-j} > K)$.

In [5]:
def Closed_CRR(S0,K,u,d,r,p,n,T):
  num_up = np.arange(0,n+1)
  St = S0*(u**num_up)*(d**(n-num_up))
  a = np.searchsorted(St,K,side="right")
  discount = np.exp(-r * T)
  price = np.sum([sc.binom.pmf(i,n,p) * (S0 * (u**i) * d**(n-i) - K) for i in range(a,n+1)])
  return discount * price

# Pricing Call with Closed Form Cox-Ross-Rubinstein 

In [6]:
print(f"Closed form call price: ${Closed_CRR(S0,K,u,d,r,p,n,T):,.4f}")

Closed form call price: $8.4451
